# Generate and Visualize Word Embeddings

Notes:

I am following along with my M09 Word2Vec with Novels Notes and my M09 HW as I complete this section.

## Setup

### Import Libraries

In [125]:
# import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from gensim.models import word2vec
from sklearn.manifold import TSNE as tsne

In [126]:
pio.renderers.default = 'vscode'

### Configure

In [127]:
# specify OHCO and bags
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

bags = dict(
    SENTS = OHCO[:4],
    PARAS = OHCO[:3],
    CHAPS = OHCO[:2],
    BOOKS = OHCO[:1]
)

### Load Data

In [128]:
# load in tables
LIB = pd.read_csv('data/LIB.csv', sep='\t').set_index('book_id')
CORPUS = pd.read_csv('data/CORPUS.csv', sep='\t').set_index(OHCO)
VOCAB = pd.read_csv('data/VOCAB.csv', sep='\t').set_index('term_str')

### Decide on bag and min_count

In [129]:
VOCAB['n'].describe()


count    26504.000000
mean        31.903637
std        461.206186
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max      37793.000000
Name: n, dtype: float64

In [130]:
(VOCAB['n'] < 18).sum() / len(VOCAB)

np.float64(0.8823196498641714)

I have chosen paragraph as my bag level so that the window looks at words that are part of the same narrative moment or exchange. I believe that this will show things like clue setting, interrogation scenes, revelation moments. I'm aiming to uncover thematic clustering.
(Sentence would have been too narrow of a window (but would have given interesting information about how Christie writes) and chapter would have been too broad of a window, missing contextual associations.)

I have also chosen a min_count of 18 because I want it to be high enough to output reliable enough vectors but not so high that I lose important genre-specific terms.

## Generate word embeddings

### Convert to Gensim

Now I create a Gensim-style corpus of "documents."

A corpus in Gensim is just a list of tokens by bag.

In [131]:
# create gensim corpus of documents from CORPUS
docs = CORPUS.groupby(bags['PARAS']).term_str.apply(list).tolist()

### Use Gensim's module to generate word embeddings

Use the gensim corpus to generate a model.

In [132]:
# word2vec parameters
w2v_params = dict(
    window = 2, # default in our course is 2
    vector_size = 256,
    min_count = 18, # this limits the vocab
    workers = 1, # keep at 1 for reproducibility
    seed = 36
)

In [133]:
model = word2vec.Word2Vec(docs, **w2v_params)
# model.wv.vectors

In [134]:
# convert model to data frame
VOCAB_W2V = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key)
VOCAB_W2V.index.name = 'term_str'
VOCAB_W2V

,0,1,2,3,4,5,6,7,8,9,...,246,247,248,249,250,251,252,253,254,255
term_str,,,,,,,,,,,,,,,,,,,,,
the,-0.118302,-0.460322,0.069786,-0.464155,0.096079,0.249094,0.721907,-0.355512,0.878449,0.289354,...,-0.361407,-0.065535,-0.260855,0.227060,-0.031222,0.225419,-0.115821,-0.196077,-0.386251,0.297772
i,0.141270,-0.026270,0.553079,-0.226452,-0.235183,1.042999,-0.260095,-0.793340,-0.290901,0.533587,...,0.294585,0.796205,0.558532,-0.082435,0.091798,0.564123,-0.161112,0.481636,0.170361,0.682530
to,-0.322985,0.214674,-0.134367,-0.590932,-0.306928,0.795758,-0.542940,0.081180,0.068501,0.206932,...,0.125866,0.143710,-0.029030,0.505641,0.172421,0.479117,-0.692576,0.574984,-0.229447,0.252814
a,-0.742762,-0.550337,0.467220,-0.372738,-0.570342,-0.177543,0.589148,0.017989,-0.210485,0.368958,...,0.353016,0.615163,0.549209,-0.894247,-0.066458,-0.465575,-0.150654,-0.508063,-0.100395,0.304210
and,-0.087503,0.207163,-0.060242,0.006613,-0.001502,-0.137161,0.025501,-0.030658,-0.114538,-0.151485,...,0.061952,0.107625,-0.018368,-0.358026,-0.107565,-0.018066,-0.063784,0.096078,0.148123,0.252840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
squirrel,-0.022037,0.021660,-0.076836,0.002048,-0.070267,0.055639,0.082649,0.045713,0.029820,-0.061462,...,0.101815,-0.027826,0.056801,-0.079866,0.039340,0.000034,-0.047477,-0.011315,-0.040775,0.138403
upper,-0.005744,-0.023118,-0.090902,-0.015249,-0.099739,0.032617,0.166531,0.034085,0.068980,-0.042883,...,0.126506,-0.001293,0.078273,-0.023223,0.034939,-0.013610,-0.046576,-0.010429,-0.089250,0.179707
venture,0.000124,-0.007507,-0.038095,-0.010426,-0.035941,0.077396,0.105521,-0.006263,0.029931,-0.058169,...,0.068332,0.031007,0.067920,-0.036053,0.041863,0.036905,-0.035573,0.008164,-0.040438,0.130476


## Save Output

In [135]:
# save the VOCAB_W2V table to csv
VOCAB_W2V.to_csv('data/VOCAB_W2V.csv', sep='\t', index=True)

## Visualize using SKLearn's tSNE library

In [136]:
PP = 20 # Try 1, 20, 100, etc.

tsne_engine = tsne(
    perplexity=PP, 
    n_components=2, 
    init='pca', 
    max_iter=2500, # iterations (more gives algorithm more time which generally gives cleaner clusters)
    random_state=36
)
TSNE = pd.DataFrame(
    tsne_engine.fit_transform(VOCAB_W2V), 
    columns=['x','y'], 
    index=VOCAB_W2V.index)

TSNE

,x,y
term_str,,
the,1.445845,57.894085
i,11.307259,-59.041180
to,54.990707,-35.744808
a,1.157103,60.245567
and,20.407442,31.015877
...,...,...
squirrel,5.368810,-11.919381
upper,-5.131276,-2.102241
venture,-5.011897,-21.189808


I just want to narrow this visual down to nouns.

In [137]:
noun_terms = VOCAB[VOCAB['max_pos_group'] == 'NN'].index
TSNE_nouns = TSNE.loc[TSNE.index.isin(noun_terms)]

In [138]:
px.scatter(TSNE_nouns.reset_index(), 'x', 'y', 
        text='term_str', 
        hover_name='term_str',  
        # size='n',
        height=1000,
        width=1200)\
    .update_traces(
        # mode='markers',
        mode='markers+text', 
        textfont=dict(color='black', size=14, family='Arial'),
        textposition='top center')

Body parts form the core of this cluster and are surrounded by clothing and accessories. Then come perceptual actions and then social performances. This structure is indicative of Agatha Christie's style of reading characters through physical observation (i.e. "a determined chin") that moves from anatomy outward to behavior.

In detective fiction that does not hide clues from the reader, the body is a surface for evidence. Suspects will betray themselves physically and witnesses describe that which they see. Poirot then reads the tells of those he interacts with.